In [1]:
import polars as pl

D = 20_000
S = 300
h = 0.15

c = [1.00, 0.96, 0.92]
breaks = [18_000, 36_000]

lo = [0] + breaks
hi = breaks + [float("inf")]

df = pl.DataFrame(
    {"category": [1, 2, 3], "from": lo, "to": hi, "c": c},
    schema={"category": pl.Int32, "from": pl.Float64, "to": pl.Float64, "c": pl.Float64},
)
df

category,from,to,c
i32,f64,f64,f64
1,0.0,18000.0,1.0
2,18000.0,36000.0,0.96
3,36000.0,inf,0.92


In [6]:
df = (
    df
    # R_i = R_{i-1} + b_{i-1}(c_{i-1} - c_i)
    .with_columns(
        (pl.col("to").shift(1) * (pl.col("c").shift(1) - pl.col("c")))
        .fill_null(0)
        .cum_sum()
        .alias("R")
    )
)

df

category,from,to,c,Q_star,feasible,R
i32,f64,f64,f64,f64,bool,f64
1,0.0,null,1.0,2362.497693,null,0.0
2,18000.0,720.0,0.96,2411.214111,false,0.0
3,36000.0,2880.0,0.92,2463.074111,false,28.8


In [7]:
df = (
    df
    # Q*_i = sqrt( S*D / (c_i * (1 + h/2)) )
    .with_columns((S * D / (pl.col("c") * (1 + h / 2))).sqrt().alias("Q_star"))
)

df

category,from,to,c,Q_star,feasible,R
i32,f64,f64,f64,f64,bool,f64
1,0.0,null,1.0,2362.497693,null,0.0
2,18000.0,720.0,0.96,2411.214111,false,0.0
3,36000.0,2880.0,0.92,2463.074111,false,28.8


In [8]:
df = (
    df
    # feasible only if Q*_i actually lands inside category i's own range
    .with_columns(
        pl.when(pl.col("to").is_infinite())
        .then(pl.col("Q_star") > pl.col("from"))
        .otherwise((pl.col("Q_star") > pl.col("from")) & (pl.col("Q_star") <= pl.col("to")))
        .alias("feasible")
    )
)

df

category,from,to,c,Q_star,feasible,R
i32,f64,f64,f64,f64,bool,f64
1,0.0,null,1.0,2362.497693,null,0.0
2,18000.0,720.0,0.96,2411.214111,false,0.0
3,36000.0,2880.0,0.92,2463.074111,false,28.8


In [9]:
df = (
    df
    .with_columns((pl.col("R") + pl.col("c") * pl.col("Q_star")).alias("C_Q"))
)
df


category,from,to,c,Q_star,feasible,R,C_Q
i32,f64,f64,f64,f64,bool,f64,f64
1,0.0,null,1.0,2362.497693,null,0.0,2362.497693
2,18000.0,720.0,0.96,2411.214111,false,0.0,2314.765546
3,36000.0,2880.0,0.92,2463.074111,false,28.8,2294.828182


In [10]:
df = (
    df
    .with_columns(
        (S * D / pl.col("Q_star") + h * pl.col("C_Q") / 2 + pl.col("C_Q")).alias("TVC")
    )
)

df

category,from,to,c,Q_star,feasible,R,C_Q,TVC
i32,f64,f64,f64,f64,bool,f64,f64,f64
1,0.0,null,1.0,2362.497693,null,0.0,2362.497693,5079.37004
2,18000.0,720.0,0.96,2411.214111,false,0.0,2314.765546,4976.745925
3,36000.0,2880.0,0.92,2463.074111,false,28.8,2294.828182,4902.920591


In [ ]:
best = df.filter(pl.col("feasible")).sort("TVC").row(0, named=True)
    
print(f"Optimal: category {best['category']}, Q* = {best['Q_star']:.1f}, TVC = {best['TVC']:.2f}")

category,from,to,c,Q_star,feasible,R,C_Q,TVC
i32,f64,f64,f64,f64,bool,f64,f64,f64
